# Task 1 — Duck vs Chicken: Transfer Learning with a Pre-trained CNN

**Goal:** Fine-tune a pre-trained convolutional neural network on a custom image dataset of ~100 ducks + ~100 chickens, then produce a classification report on a held-out test set.

**Approach**
1. **Automatically download** ~120 images of each class from Bing Image Search using `icrawler` (saves you from manually collecting images).
2. Split into train / val / test (70/15/15) while preserving class balance.
3. Load **ResNet-50** pre-trained on ImageNet, swap out its final 1000-class layer for a 2-class head.
4. **Two-phase fine-tuning**:
   - Phase 1 — Freeze the backbone, train only the classifier head (fast warm-up).
   - Phase 2 — Unfreeze the last residual stage (`layer4`) and fine-tune the whole thing at a lower LR.
5. Evaluate on test set with classification report, confusion matrix, and ROC curve.

**Why ResNet-50?** It's a strong, well-understood baseline that ships with `torchvision`, trains fast on a Colab T4, and its pretrained ImageNet features already contain excellent bird-shaped concepts.

> **Runtime:** Google Colab (GPU). Go to `Runtime → Change runtime type → T4 GPU` before running.


## Step 1 — Install dependencies & import libraries

In [ ]:
# icrawler handles the actual image scraping from Bing.
# Everything else is already in Colab.
!pip install -q icrawler

In [ ]:
import os, random, shutil, warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, models
from torchvision.datasets import ImageFolder

from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc
)

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU   : {torch.cuda.get_device_name(0)}")

## Step 2 — Download ~120 images per class from Bing

We ask `icrawler` for 120 images per class so that after ~20% duds/duplicates are dropped by the automatic filter, we land at roughly 100 usable images per class. If you already have your own images, skip this cell and just drop them into `data/chicken` and `data/duck`.


In [ ]:
from icrawler.builtin import BingImageCrawler

RAW_DIR = Path("data_raw")
RAW_DIR.mkdir(exist_ok=True)

# Use multiple search terms per class to improve diversity
download_plan = {
    "chicken": ["chicken bird", "hen standing", "rooster outdoors"],
    "duck":    ["duck bird", "mallard duck", "duck on water"],
}

IMGS_PER_QUERY = 40   # 3 queries × 40 = 120 candidates per class

for cls, queries in download_plan.items():
    cls_dir = RAW_DIR / cls
    cls_dir.mkdir(exist_ok=True)
    for q in queries:
        print(f"Downloading '{q}' → {cls_dir}")
        crawler = BingImageCrawler(
            storage={"root_dir": str(cls_dir)},
            log_level=40,            # quiet
        )
        crawler.crawl(
            keyword=q,
            max_num=IMGS_PER_QUERY,
            file_idx_offset="auto",  # avoids name clashes between queries
            min_size=(150, 150),
        )

# Quick report
for cls in download_plan:
    n = len(list((RAW_DIR / cls).iterdir()))
    print(f"{cls:8s}: {n} images downloaded")

### Filter out corrupt / non-image files

Web scrapers often pull in broken or non-image files. We validate each download by trying to open it with PIL; any file that fails is deleted.

In [ ]:
def drop_bad_images(folder):
    """Remove files that PIL can't decode or that are too small."""
    removed = 0
    for p in Path(folder).iterdir():
        try:
            with Image.open(p) as im:
                im.verify()                        # integrity check
            with Image.open(p) as im:              # verify closes the file
                im = im.convert("RGB")
                if min(im.size) < 100:             # too small to be useful
                    raise ValueError("too small")
        except Exception:
            p.unlink(missing_ok=True)
            removed += 1
    return removed

for cls in download_plan:
    n_removed = drop_bad_images(RAW_DIR / cls)
    n_kept    = len(list((RAW_DIR / cls).iterdir()))
    print(f"{cls:8s}: removed {n_removed:3d}, kept {n_kept:3d}")

## Step 3 — Create train / val / test folders (70 / 15 / 15)

We copy files into an `organised/` directory with a standard `train/<class>/*.jpg` layout so we can plug straight into `torchvision.datasets.ImageFolder`.

In [ ]:
ORG_DIR = Path("organised")
if ORG_DIR.exists():
    shutil.rmtree(ORG_DIR)         # clean rebuild if re-running

SPLIT_RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}

for cls in download_plan:
    files = sorted((RAW_DIR / cls).iterdir())
    random.Random(SEED).shuffle(files)

    n = len(files)
    n_train = int(n * SPLIT_RATIOS["train"])
    n_val   = int(n * SPLIT_RATIOS["val"])

    buckets = {
        "train": files[:n_train],
        "val":   files[n_train:n_train + n_val],
        "test":  files[n_train + n_val:],
    }
    for split, paths in buckets.items():
        dst = ORG_DIR / split / cls
        dst.mkdir(parents=True, exist_ok=True)
        for p in paths:
            shutil.copy(p, dst / p.name)

# Show the resulting counts
print(f"{'split':<8}{'chicken':>10}{'duck':>8}")
for split in ["train", "val", "test"]:
    c = len(list((ORG_DIR / split / "chicken").iterdir()))
    d = len(list((ORG_DIR / split / "duck").iterdir()))
    print(f"{split:<8}{c:>10}{d:>8}")

## Step 4 — Data transforms

Because our dataset is tiny (~200 total images), we lean heavily on augmentation for the training set to artificially expand it. The validation / test sets only get a deterministic resize + normalisation.

Values `MEAN` and `STD` are the standard ImageNet stats — required since we're using ImageNet pre-trained weights.

In [ ]:
IMG_SIZE = 224                         # ResNet-50's native input size
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.1)),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

## Step 5 — Datasets & DataLoaders

In [ ]:
BATCH_SIZE = 16

train_ds = ImageFolder(ORG_DIR / "train", transform=train_tf)
val_ds   = ImageFolder(ORG_DIR / "val",   transform=eval_tf)
test_ds  = ImageFolder(ORG_DIR / "test",  transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

CLASS_NAMES = train_ds.classes
print(f"Classes      : {CLASS_NAMES}  (index map: {train_ds.class_to_idx})")
print(f"Train images : {len(train_ds)}")
print(f"Val   images : {len(val_ds)}")
print(f"Test  images : {len(test_ds)}")

## Step 6 — Visualise a batch of augmented training images

In [ ]:
def denormalize(t):
    m = torch.tensor(MEAN).view(3, 1, 1)
    s = torch.tensor(STD).view(3, 1, 1)
    return (t * s + m).clamp(0, 1)

imgs, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
fig.suptitle("Training batch (after augmentation)", fontweight="bold")
for ax, img, lb in zip(axes.flat, imgs, labels):
    ax.imshow(denormalize(img).permute(1, 2, 0).numpy())
    ax.set_title(CLASS_NAMES[lb], fontsize=10)
    ax.axis("off")
plt.tight_layout(); plt.show()

## Step 7 — Load pre-trained ResNet-50 and replace its head

`torchvision.models.resnet50(weights=...)` downloads the ImageNet-pretrained weights. We swap `model.fc` (which outputs 1000 ImageNet classes) for a fresh 2-neuron linear layer for our duck/chicken task.

**Initial freezing strategy:** we freeze every parameter except the new `fc` layer. This means only the classifier head is trained in Phase 1.

In [ ]:
def build_model(num_classes=2):
    weights = models.ResNet50_Weights.IMAGENET1K_V2
    model = models.resnet50(weights=weights)

    # Freeze the whole backbone
    for p in model.parameters():
        p.requires_grad = False

    # Replace the 1000-way head with a 2-way head (unfrozen by default)
    in_feats = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_feats, num_classes),
    )
    return model


def count_params(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   Trainable: {trainable:>12,}  /  Total: {total:,}  "
          f"({100*trainable/total:.2f}%)")


model = build_model(num_classes=len(CLASS_NAMES)).to(DEVICE)
print("Phase-1 parameter counts (only head should be trainable):")
count_params(model)

## Step 8 — Training / evaluation helpers

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    """Single pass. If optimizer is None, runs in eval mode."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            loss   = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            correct    += (logits.argmax(1) == labels).sum().item()
            total      += labels.size(0)
    return total_loss / total, 100 * correct / total


history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

def train_phase(name, epochs, optimizer, criterion, best_state):
    """Run a phase and track history. Saves best weights (lowest val loss)."""
    print(f"\n─── {name} ({epochs} epochs) ───")
    best_loss = best_state["best_val_loss"]
    for ep in range(1, epochs + 1):
        tr_l, tr_a = run_epoch(model, train_loader, criterion, optimizer)
        va_l, va_a = run_epoch(model, val_loader,   criterion)

        history["train_loss"].append(tr_l); history["train_acc"].append(tr_a)
        history["val_loss"].append(va_l);   history["val_acc"].append(va_a)

        tag = ""
        if va_l < best_loss:
            best_loss = va_l
            best_state["best_val_loss"] = va_l
            best_state["weights"] = {k: v.detach().cpu().clone()
                                     for k, v in model.state_dict().items()}
            tag = "  ← new best"
        print(f"  Epoch {ep:2d} | train loss {tr_l:.4f} acc {tr_a:5.2f}%  "
              f"| val loss {va_l:.4f} acc {va_a:5.2f}%{tag}")

## Step 9 — Phase 1: Train the classifier head only

The backbone is frozen, so forward passes re-use ImageNet features and the only thing that updates is our new `fc` layer. This is fast and gives us a sane initialisation for the head before we touch the rest of the network.

In [ ]:
criterion = nn.CrossEntropyLoss()

PHASE1_EPOCHS = 8
PHASE1_LR     = 1e-3

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=PHASE1_LR, weight_decay=1e-4,
)

best_state = {"best_val_loss": float("inf"), "weights": None}
train_phase("Phase 1 — head only", PHASE1_EPOCHS, optimizer, criterion, best_state)

## Step 10 — Phase 2: Unfreeze `layer4` and fine-tune with a small LR

ResNet-50 is made of 4 residual stages (`layer1` → `layer4`). The later stages contain more domain-specific features. Unfreezing *just* `layer4` + the head strikes a balance: the model can specialise its high-level features for birds without destroying the generic edge/texture detectors in `layer1`–`layer3`.

We drop the learning rate by 5× so the pre-trained features aren't overwritten too aggressively.

In [ ]:
# Unfreeze the last residual stage
for p in model.layer4.parameters():
    p.requires_grad = True

print("Phase-2 parameter counts:")
count_params(model)

PHASE2_EPOCHS = 10
PHASE2_LR     = 2e-4

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=PHASE2_LR, weight_decay=1e-4,
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=PHASE2_EPOCHS, eta_min=PHASE2_LR * 0.05,
)

# We'll step the scheduler manually since train_phase doesn't know about it
for ep in range(1, PHASE2_EPOCHS + 1):
    tr_l, tr_a = run_epoch(model, train_loader, criterion, optimizer)
    va_l, va_a = run_epoch(model, val_loader,   criterion)
    scheduler.step()

    history["train_loss"].append(tr_l); history["train_acc"].append(tr_a)
    history["val_loss"].append(va_l);   history["val_acc"].append(va_a)

    tag = ""
    if va_l < best_state["best_val_loss"]:
        best_state["best_val_loss"] = va_l
        best_state["weights"] = {k: v.detach().cpu().clone()
                                 for k, v in model.state_dict().items()}
        tag = "  ← new best"
    print(f"  Epoch {ep:2d} | train loss {tr_l:.4f} acc {tr_a:5.2f}%  "
          f"| val loss {va_l:.4f} acc {va_a:5.2f}%  "
          f"| lr {scheduler.get_last_lr()[0]:.2e}{tag}")

# Restore the best weights found across both phases
model.load_state_dict(best_state["weights"])
print(f"\nRestored best weights (val loss = {best_state['best_val_loss']:.4f}).")

## Step 11 — Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

epochs = range(1, len(history["train_loss"]) + 1)
phase_boundary = PHASE1_EPOCHS + 0.5

axes[0].plot(epochs, history["train_loss"], "o-", label="train")
axes[0].plot(epochs, history["val_loss"],   "s-", label="val")
axes[0].axvline(phase_boundary, color="gray", ls="--", alpha=0.5, label="phase boundary")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history["train_acc"], "o-", label="train")
axes[1].plot(epochs, history["val_acc"],   "s-", label="val")
axes[1].axvline(phase_boundary, color="gray", ls="--", alpha=0.5, label="phase boundary")
axes[1].set_title("Accuracy (%)"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

## Step 12 — Classification report on the test set

In [ ]:
@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE)
        probs = torch.softmax(model(imgs), dim=1).cpu().numpy()
        y_true.extend(labels.numpy())
        y_pred.extend(probs.argmax(1))
        y_prob.extend(probs)
    return np.array(y_true), np.array(y_pred), np.array(y_prob)


y_true, y_pred, y_prob = collect_predictions(model, test_loader)

print("=" * 60)
print("CLASSIFICATION REPORT — Test Set")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

## Step 13 — Confusion matrix & ROC curve

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title("Confusion matrix (counts)")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")

# ROC curve (class 1 = duck as positive, arbitrary)
pos_idx = 1
fpr, tpr, _ = roc_curve(y_true == pos_idx, y_prob[:, pos_idx])
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc:.4f}")
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_xlabel("False positive rate")
axes[1].set_ylabel("True positive rate")
axes[1].set_title(f"ROC curve (positive = {CLASS_NAMES[pos_idx]})")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f"AUC-ROC: {roc_auc:.4f}")

## Step 14 — Inspect misclassified test images

In [ ]:
miss = np.where(y_pred != y_true)[0]
print(f"Misclassified: {len(miss)} / {len(y_true)}  "
      f"({100 * len(miss) / len(y_true):.1f}%)")

if len(miss):
    test_paths = [p for p, _ in test_ds.samples]
    n = min(8, len(miss))
    fig, axes = plt.subplots(1, n, figsize=(2.5 * n, 3))
    if n == 1: axes = [axes]
    for ax, idx in zip(axes, miss[:n]):
        img = Image.open(test_paths[idx]).convert("RGB").resize((180, 180))
        ax.imshow(img)
        conf = y_prob[idx, y_pred[idx]] * 100
        ax.set_title(f"T:{CLASS_NAMES[y_true[idx]]}\n"
                     f"P:{CLASS_NAMES[y_pred[idx]]} ({conf:.0f}%)",
                     fontsize=9, color="crimson")
        ax.axis("off")
    plt.suptitle("Misclassified test images", fontweight="bold")
    plt.tight_layout(); plt.show()
else:
    print("Perfect classification on the test set.")

## Step 15 — Final summary

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

acc  = accuracy_score(y_true, y_pred) * 100
prec = precision_score(y_true, y_pred, average="weighted") * 100
rec  = recall_score(y_true, y_pred, average="weighted") * 100
f1   = f1_score(y_true, y_pred, average="weighted") * 100

print("=" * 55)
print("  FINAL RESULTS — Duck vs Chicken")
print("=" * 55)
print(f"  Backbone            : ResNet-50 (ImageNet V2 weights)")
print(f"  Fine-tuning         : 2 phases (head only → +layer4)")
print(f"  Total epochs        : {len(history['train_loss'])}")
print(f"  Best val loss       : {best_state['best_val_loss']:.4f}")
print(f"  Test accuracy       : {acc:.2f}%")
print(f"  Weighted precision  : {prec:.2f}%")
print(f"  Weighted recall     : {rec:.2f}%")
print(f"  Weighted F1-score   : {f1:.2f}%")
print(f"  AUC-ROC             : {roc_auc:.4f}")
print("=" * 55)